In [3]:
import sys
import os
import subprocess

print("Python executable:", sys.executable)
print("Python prefix:", sys.prefix)
print("Conda environment:", os.environ.get('CONDA_DEFAULT_ENV'))
print("Conda prefix:", os.environ.get('CONDA_PREFIX'))

# Check pip location
result = subprocess.run(['which', 'pip'], capture_output=True, text=True)
print("Pip location:", result.stdout.strip())

# Check pip version
result = subprocess.run(['pip', '--version'], capture_output=True, text=True)
print("Pip version:", result.stdout.strip())

# Add environment's bin directory to PATH
env_bin = os.path.dirname(sys.executable)
current_path = os.environ['PATH']
if env_bin not in current_path:
    os.environ['PATH'] = f"{env_bin}:{current_path}"

# Verify fix
result = subprocess.run(['which', 'pip'], capture_output=True, text=True)
print("Fixed pip location:", result.stdout.strip())

Python executable: /home/sagemaker-user/.conda/envs/venv-LLMsFromScratch/bin/python3
Python prefix: /home/sagemaker-user/.conda/envs/venv-LLMsFromScratch
Conda environment: base
Conda prefix: /opt/conda
Pip location: /opt/conda/bin/pip
Pip version: pip 25.1.1 from /opt/conda/lib/python3.12/site-packages/pip (python 3.12)
Fixed pip location: /home/sagemaker-user/.conda/envs/venv-LLMsFromScratch/bin/pip


In [4]:
!pwd

/home/sagemaker-user/LLMsFromScratch/Chapter_2


# Code

In [5]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total Characters:", len(raw_text))

Total Characters: 20479


In [6]:
print(raw_text[:99])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


### Split text

In [7]:
import re
text = "Kya haal chal bro? sab sahi hai?"

result = re.split(r'(\s)', text)

print(result)

['Kya', ' ', 'haal', ' ', 'chal', ' ', 'bro?', ' ', 'sab', ' ', 'sahi', ' ', 'hai?']


In [8]:
# spliting punctuation too

result = re.split(r'([,.?]|\s)', text)

print(result)

['Kya', ' ', 'haal', ' ', 'chal', ' ', 'bro', '?', '', ' ', 'sab', ' ', 'sahi', ' ', 'hai', '?', '']


In [9]:
# removing spaces

result = [ item for item in result if item.strip() ]

print(result)

['Kya', 'haal', 'chal', 'bro', '?', 'sab', 'sahi', 'hai', '?']


In [10]:
# test it all 

text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]

print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [11]:
# applying the tokenizer to complete text file

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)

preprocessed = [item.strip() for item in preprocessed if item.strip()]

print(len(preprocessed))

4690


In [12]:
# I first got around 9000 token - forgot removed the spaces.
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


### Converting to Token ID

In [13]:
# counts the unique token 
all_words = sorted(set(preprocessed))

vocab_size = len(all_words)

vocab_size

1130

In [14]:
# Creating a vocabulary

vocab = { token:integer for integer,token in enumerate(all_words) }

# vocab 

for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


### Implementing a simple text tokenizer

In [15]:
'''
#1 Stores the vocabulary as a class attribute for access in the encode and decode methods
#2 Creates an inverse vocabulary that maps token IDs back to the original text tokens
#3 Processes input text into token IDs
#4 Converts token IDs back into text
#5 Removes spaces before the specified punctuation 
'''

class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items() }

    def encode(self,text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [ item.strip() for item in preprocessed if item.strip() ]
        ids = [self.str_to_int[s] for s in preprocessed]

        return ids

    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)

        return text

In [16]:
# calling the function 

tokenizer = SimpleTokenizerV1(vocab) # passing the mapping of word to token id

text =  """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""

ids = tokenizer.encode(text)

print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [17]:
# Printing back the words from token id - this is what happens at the end of LLM output just before you see the output

print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [18]:
# calling the function on words that are not stored in the tokenizer dicitionary - common key value error

hinid_test_tokenizer = SimpleTokenizerV1(vocab) # passing the mapping of word to token id

text =  """"kya haal chal bhai"""

ids = hinid_test_tokenizer.encode(text)

print(ids)


KeyError: 'kya'

### Adding Custom Token

In [19]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}             

print(len(vocab.items()))

1132


In [20]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


### Simple text tokenizer that handles unknown words

In [21]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [ item.strip() for item in preprocessed if item.strip() ]
        preprocessed = [ item if item in self.str_to_int else "<|unk|>" for item in preprocessed]

        ids = [self.str_to_int[s] for s in preprocessed]
        
        return ids

    def decode(self,ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [22]:
# this is how they combine large document to tell the model that this is sepearate doc

text1 = "Hello, do you like tea?"

text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [23]:
tokenizerv2 = SimpleTokenizerV2(vocab)
print(tokenizerv2.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [24]:
print(tokenizerv2.decode(tokenizerv2.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


### Byte pair encoding

In [25]:
%%capture
!pip install tiktoken

In [26]:
# Version 
from importlib.metadata import version
import tiktoken

print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.12.0


In [27]:
bpe_tokenizer = tiktoken.get_encoding("gpt2")

In [28]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = bpe_tokenizer.encode(text, allowed_special= {"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [29]:
print(bpe_tokenizer.decode(integers))

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [30]:
text = "Swapnil"

encode_swapnil = bpe_tokenizer.encode(text, allowed_special= {"<|endoftext|>"})

encode_swapnil

[10462, 499, 45991]

In [31]:
print(bpe_tokenizer.decode([45991]))

nil


### Data sampling with a sliding window

In [32]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = bpe_tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [33]:
# removing first 50 token from passage - optional 
enc_sample = enc_text[50:]

In [34]:
# Creating context size - The context size determines how many tokens are included in the input. 
context_size = 5

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print("x: ", x)
print("y:      ", y)  

x:  [290, 4920, 2241, 287, 257]
y:       [4920, 2241, 287, 257, 4489]


In [35]:
for i in range(1, context_size):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "----->", desired)

[290] -----> 4920
[290, 4920] -----> 2241
[290, 4920, 2241] -----> 287
[290, 4920, 2241, 287] -----> 257


In [36]:
for i in range(1, context_size):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(bpe_tokenizer.decode(context), "----->", bpe_tokenizer.decode([desired]))

 and ----->  established
 and established ----->  himself
 and established himself ----->  in
 and established himself in ----->  a


### A dataset for batched inputs and targets

In [66]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetv1(Dataset): # Class inheretence
    def __init__(self, txt, bpe_tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = bpe_tokenizer.encode(txt) # Tokenized entire text

        for i in range(0, len(token_ids) - max_length, stride): # chuck the corpus into fixed size of max length with no of skiiping sliding token decide by stride
            input_chunks = token_ids[i:i + max_length]
            target_chunks = token_ids[i+1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunks))
            self.target_ids.append(torch.tensor(target_chunks))

    #3 Returns the total number of rows in the dataset 
    def __len__(self): # magic function - inhereted from parent class
        return len(self.input_ids)

    #4 Returns a single row from the dataset 
    def __getitem__(self, idx): # magic function - inhereted from parent class
        return self.input_ids[idx], self.target_ids[idx]
        

In [67]:
def create_dataloader_v1( txt, batch_size = 4, max_length=256, stride=128, shuffle=True, drop_last = True, num_workers = 0): #3 drop_last=True drops the last batch if it is shorter than the specified batch_size to prevent loss spikes during training. 
    tokenizer= tiktoken.get_encoding("gpt2") # initialize the tokenzier 
    
    dataset = GPTDatasetv1(txt, tokenizer, max_length, stride)
    
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers) #4 num_workers = The number of CPU processes to use for preprocessing 
    
    return dataloader

In [68]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(raw_text, batch_size=1, shuffle=True)
data_iter = iter(dataloader)  #1 Converts dataloader into a Python iterator to fetch the next entry via Python’s built-in next() function 
first_batch = next(data_iter)
print(first_batch)

[tensor([[  284,  1234,  8737,   656, 19133,   553,   373,   530,   286,   262,
          7877,    72,  3150,   339,  8104,   866,  1973,   262, 37918,   411,
           290,  8465,   286,   281, 33954,   271,  3973,  9899, 14678, 40556,
            12, 11487,    11,   618,    11,   319,   257,  1568,  1110,    11,
           314,   550,   757,  1057,   625,   422, 22489, 40089,    26,   290,
          9074,    13,   402,   271, 10899,    11,   307,  3723,   319,   683,
            11,  2087,   329,   616, 35957,    25,   366, 14295,   318,   523,
         34813,   306,  8564,   284,   790,  1296,   286,  8737,   526,   198,
           198, 43920,  3619,     0,   632,   550,  1464,   587,   465, 10030,
           284,   423,  1466,   910,   884,  1243,   286,   683,    25,   262,
          1109,   815,   307,   900,   866,   287,  1070,   268,  2288,    13,
          1867,  7425,   502,   783,   373,   326,    11,   329,   262,   717,
           640,    11,   339,   581,  4714,   262, 

### Data loaders with different strides and context sizes

In [70]:
dataloader2 = create_dataloader_v1(raw_text, batch_size=1, max_length=8, stride=1, shuffle= True)
data_iter = iter(dataloader2)
first_batch = next(data_iter)
print(first_batch)

[tensor([[2474,  198,  198, 1537,   11,  351,  262, 3960]]), tensor([[ 198,  198, 1537,   11,  351,  262, 3960,  319]])]


## Creating token embeddings

In [72]:
vocab_size = 6

output_dim = 3

torch.manual_seed(123)

embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [74]:
print(embedding_layer(torch.tensor([0])))

tensor([[ 0.3374, -0.1778, -0.1690]], grad_fn=<EmbeddingBackward0>)


In [86]:
input_ids = torch.tensor([0, 1, 2, 3, 3, 4, 5])
input_ids

tensor([0, 1, 2, 3, 3, 4, 5])

In [87]:
print(embedding_layer(torch.tensor(input_ids)))

tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], grad_fn=<EmbeddingBackward0>)


/tmp/ipykernel_2958/3846183200.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  print(embedding_layer(torch.tensor(input_ids)))


## Encoding word positions

In [102]:
max_length = 4
dataloader2 = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=1, shuffle= True)
data_iter = iter(dataloader2)
inputs, targets = next(data_iter)
print("Input Token IDs : \n", inputs, "\n\n",  "Target Token IDs : \n", targets)
print("\nInputs shape:\n", inputs.shape)

Input Token IDs : 
 tensor([[  338,  3451,   286,  5287],
        [28060,     6,   416,   617],
        [   13,   366,  1858,   547],
        [ 1114,  9074,    13,   402],
        [   11,   965,  2860,  1359],
        [  339,  4808,  9776,    62],
        [  284,  1986,   262,  1109],
        [  757, 13984,   198,   198]]) 

 Target Token IDs : 
 tensor([[ 3451,   286,  5287,   852],
        [    6,   416,   617,   530],
        [  366,  1858,   547,  1528],
        [ 9074,    13,   402,   271],
        [  965,  2860,  1359,   290],
        [ 4808,  9776,    62,  1364],
        [ 1986,   262,  1109,   351],
        [13984,   198,   198,  5779]])

Inputs shape:
 torch.Size([8, 4])


In [103]:
vocab_size = 50257

output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [104]:
token_embeddings = token_embedding_layer(inputs)

print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [109]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)
print(pos_embeddings)

torch.Size([4, 256])
tensor([[ 0.9591,  0.6523, -0.1369,  ...,  0.9495, -1.7711,  0.7002],
        [-0.1813,  0.8548, -0.7214,  ..., -0.0076,  0.3891,  2.0707],
        [ 1.3912,  1.0278, -0.8874,  ..., -2.0360, -2.2768,  1.4118],
        [ 0.1460, -1.0633, -0.3271,  ...,  0.5971, -0.6039,  0.8326]],
       grad_fn=<EmbeddingBackward0>)


In [112]:
input_embeddings = token_embeddings + pos_embeddings

print(input_embeddings.shape)
print(input_embeddings)

torch.Size([8, 4, 256])
tensor([[[ 3.2816, -0.8385, -1.5988,  ...,  0.4515, -2.2307, -0.4862],
         [-1.8250,  1.0060, -2.1372,  ...,  0.0839, -2.0361,  2.8271],
         [ 0.6669,  2.1464, -1.1817,  ..., -1.0652, -3.3665, -0.5665],
         [ 1.9827,  0.2662,  0.8666,  ...,  1.5479, -0.2560,  1.5300]],

        [[ 1.5635,  1.2202,  0.7103,  ...,  1.8201, -1.7266, -0.3727],
         [-0.9123,  2.2052, -0.1703,  ...,  2.0441,  0.3794,  1.1910],
         [ 0.0512,  3.1379, -1.4317,  ..., -3.2079, -3.1840, -0.3043],
         [-0.4827, -2.8640, -0.8916,  ...,  1.5868, -1.0954,  2.1030]],

        [[ 1.2778,  0.6395, -0.9184,  ...,  1.1015, -2.2767,  1.0639],
         [ 1.1060,  0.8302, -0.0743,  ...,  0.1004, -0.5003,  0.5185],
         [ 1.6672,  1.3556, -0.1430,  ..., -3.8121, -3.3954,  0.8257],
         [-0.3197, -1.1752, -0.6899,  ..., -0.1317, -1.9918,  0.2513]],

        ...,

        [[ 2.6909, -1.0160,  0.4394,  ...,  1.0998, -1.1042,  0.9741],
         [-0.1979,  0.0602, -0.32